# Submission 3 — Project Data, Streaming & Aggregation (5%)

**Course:** RBB2013 Digital Twin — May 2026
**Group project — SmartClean Twin:** Digital Twin of a mobile cleaning robot (topic 2)
**Team Members:**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |
**Repository:** https://github.com/KAI-UTP/smartclean-twin

> Live cells require `docker compose up -d` (8 containers). All outputs are
> pre-executed and saved, so the evidence is visible without running.


## 1. Streaming architecture

- **Transport:** MQTT (Eclipse Mosquitto), port 1883, JSON payloads, QoS 1
- **Rate:** 1 telemetry message/second (16 sensor fields per message)
- **Topic namespace** (multi-robot ready — robot id in every topic):

| Topic | Producer → Consumer(s) |
|---|---|
| `smartclean/SCR01/telemetry/raw` | simulator → ingestion |
| `smartclean/SCR01/telemetry/validated` | ingestion → state-engine, ai-service |
| `smartclean/SCR01/state` | state-engine → ai-service, dashboards |
| `smartclean/SCR01/prediction` | ai-service → subscribers |
| `smartclean/SCR01/alert` | state-engine → subscribers |
| `smartclean/SCR01/command/#` | command-api → simulator |
| `smartclean/SCR01/ack` | simulator → command-api |

Full payload schemas per topic: `docs/api-contract.md`.

## 2. Validation before storage

Every raw message is validated against a Pydantic schema (field presence,
types, physical ranges — e.g. battery 0–100%, temperature −10–120 °C).
Invalid messages are rejected and counted, never stored. Schema:
`shared/smartclean_common/models.py` (single source of truth for all services).

## 3. Storage

InfluxDB 2.7 time-series database — measurements: `robot_telemetry`,
`robot_state`, `robot_prediction`, `robot_alert`. Persistence across
restarts proven by `tests/system/test_persistence.py` (see Submission 4/5).


## 4. Live evidence — streaming rate

In [1]:
import json, time, urllib.request

INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    req = urllib.request.Request(INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux", "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.read().decode()

def show_last(measurement, range_s=30):
    q = (f'from(bucket: "smartclean_twin") |> range(start: -{range_s}s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    n = 0
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")
            n += 1
    if n == 0:
        print("  (no data in window — is docker compose up?)")

print("Helpers loaded.")


Helpers loaded.


In [2]:
q = ('from(bucket: "smartclean_twin") |> range(start: -60s) '
     '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "battery_soc") '
     '|> count()')
for line in flux_query(q).splitlines():
    p = line.split(",")
    if len(p) > 6 and p[1] == "_result":
        print(f"Telemetry points stored in the last 60 s: {p[6]}  (~1/second)")


Telemetry points stored in the last 60 s: battery_soc  (~1/second)


## 5. Aggregation — windowed queries (Flux)

The dashboard's *Statistical Trends* section uses server-side aggregation:
30 s mean motor current, 30 s max temperature, 1 m mean SoC, alarm counts
per minute, and a derivative() for battery discharge rate. Live examples:


In [3]:
queries = {
  "30s mean motor current (A)":
    'from(bucket: "smartclean_twin") |> range(start: -5m) '
    '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "motor_current_a") '
    '|> aggregateWindow(every: 30s, fn: mean, createEmpty: false) |> last()',
  "30s max motor temperature (C)":
    'from(bucket: "smartclean_twin") |> range(start: -5m) '
    '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "motor_temperature_c") '
    '|> aggregateWindow(every: 30s, fn: max, createEmpty: false) |> last()',
  "1m mean battery SoC (%)":
    'from(bucket: "smartclean_twin") |> range(start: -5m) '
    '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "battery_soc") '
    '|> aggregateWindow(every: 1m, fn: mean, createEmpty: false) |> last()',
  "battery discharge rate (%/min, derivative)":
    'from(bucket: "smartclean_twin") |> range(start: -5m) '
    '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "battery_soc") '
    '|> derivative(unit: 1m, nonNegative: false) |> last()',
}
for name, q in queries.items():
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 6 and p[1] == "_result":
            print(f"{name:45s} = {float(p[6]):.4f}")
            break


30s mean motor current (A)                    = 2.4600
30s max motor temperature (C)                 = 29.3900
1m mean battery SoC (%)                       = 99.8356
battery discharge rate (%/min, derivative)    = 0.0000


## 6. Rolling in-memory aggregation

The AI service additionally keeps 90-sample rolling windows in memory to
compute live rates of change (battery discharge %/min, coverage %/min) which
power the minutes-to-empty and minutes-to-finish forecasts — an example of
stream processing on top of stored aggregation.


In [4]:
print("Trend-based forecast fields (from rolling aggregation):")
q = ('from(bucket: "smartclean_twin") |> range(start: -30s) '
     '|> filter(fn: (r) => r._measurement == "robot_prediction" and '
     '(r._field == "minutes_to_empty" or r._field == "minutes_to_finish")) |> last()')
for line in flux_query(q).splitlines():
    p = line.split(",")
    if len(p) > 7 and p[1] == "_result":
        print(f"  {p[7]:22s} = {p[6]}")


Trend-based forecast fields (from rolling aggregation):
  minutes_to_empty       = 1308.5
  minutes_to_finish      = 1.7
